# Nemotron Reasoning — Holdout Eval (offline boxed-accuracy)

Evaluates a trained LoRA adapter on a **held-out slice of `train.csv`** (which has gold answers) so you get an accuracy number **without spending a daily submission**.

**Setup (Kaggle UI):** Accelerator = **GPU RTX PRO 6000**, Internet = **OFF**. Attach: the competition, the model `metric/nemotron-3-nano-30b-a3b-bf16`, the `ryanholbrook/nvidia-utility-script`, and **the adapter to eval** (attach the training kernel's output, or upload the adapter as a dataset — anything that puts `adapter_config.json` under `/kaggle/input/`).

**Clean signal:** train with the same `SEED`/`HOLDOUT_N` and `HOLDOUT_N>0` so these rows were *excluded* from training. (The first full run trained on all rows, so its holdout number is optimistic.)

## Config

In [ ]:
SEED = 42
HOLDOUT_N = 200  # number of held-out rows to evaluate (last-N after seeded shuffle)
MAX_NEW_TOKENS = 512
print(f"SEED={SEED} | HOLDOUT_N={HOLDOUT_N} | MAX_NEW_TOKENS={MAX_NEW_TOKENS}")

## 1. Load the held-out slice of train.csv (seeded, last N)

In [ ]:
import glob
import random

try:
    import polars as pl

    _tr = pl.read_csv(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
    rows = [
        {"id": str(i), "prompt": p, "answer": str(a)}
        for i, p, a in zip(
            _tr["id"].to_list(),
            _tr["prompt"].to_list(),
            _tr["answer"].to_list(),
            strict=False,
        )
    ]
except Exception:
    import pandas as pd

    _tr = pd.read_csv(
        glob.glob("/kaggle/input/**/train.csv", recursive=True)[0], dtype=str
    )
    rows = _tr.to_dict("records")

order = list(rows)
random.Random(SEED).shuffle(order)
holdout = order[-HOLDOUT_N:]
print(f"holdout rows: {len(holdout)} (of {len(rows)})")
print(
    "example:", holdout[0]["prompt"][:120], "-> answer", repr(str(holdout[0]["answer"]))
)

## 2. Setup + load base model (same fixes as the train notebook)

In [ ]:
# Setup: vendored cutlass + executable Triton ptxas (Blackwell) BEFORE importing mamba_ssm.
import glob
import os
import shutil
import site

# 1. cutlass (auto-discover; mount uses underscores: nvidia_utility_script)
cands = sorted(
    set(
        glob.glob("/kaggle/usr/lib/**/python_packages", recursive=True)
        + glob.glob("/kaggle/input/**/python_packages", recursive=True)
    )
)
print("python_packages dirs:", cands)
for c in cands:
    site.addsitedir(c)
for h in glob.glob(
    "/kaggle/**/nvidia_cutlass_dsl/python_packages/cutlass/__init__.py", recursive=True
):
    site.addsitedir(os.path.dirname(os.path.dirname(h)))

# 2. Copy vendored Triton nvidia bin to a writable+exec dir and point Triton at it via env
#    (the env override is read by the vendored Triton's knobs; verified working).
ptxas_hits = glob.glob(
    "/kaggle/usr/lib/**/triton/backends/nvidia/bin/ptxas-blackwell", recursive=True
)
print("ptxas-blackwell candidates:", ptxas_hits)
if ptxas_hits:
    srcbin = os.path.dirname(ptxas_hits[0])
    dstbin = "/tmp/triton_nvidia_bin"
    os.makedirs(dstbin, exist_ok=True)
    for b in glob.glob(os.path.join(srcbin, "*")):
        d = os.path.join(dstbin, os.path.basename(b))
        try:
            shutil.copy(b, d)
            os.chmod(d, 0o755)
        except OSError as e:
            print("copy skip:", b, e)
    wp = os.path.join(dstbin, "ptxas-blackwell")
    for var in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH"):
        os.environ[var] = wp
    os.environ["PATH"] = dstbin + ":" + os.environ.get("PATH", "")
    print("ptxas (writable):", wp, "executable:", os.access(wp, os.X_OK))

import kagglehub
import mamba_ssm  # noqa: F401  (registers Mamba CUDA kernels)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 3. The vendored Triton's NvidiaTool dataclass is unhashable (eq=True, no __hash__) and gets
#    hashed during Blackwell kernel compilation -> add a __hash__ so backward() can compile.
try:
    from triton import knobs

    if getattr(knobs.NvidiaTool, "__hash__", None) is None:
        knobs.NvidiaTool.__hash__ = lambda self: hash(getattr(self, "path", id(self)))
        print("patched NvidiaTool.__hash__")
    print("ptxas_blackwell knob:", knobs.nvidia.ptxas_blackwell.path)
except Exception as ex:
    print("triton knob inspect failed:", repr(ex))

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print("MODEL_PATH:", MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded")

## 3. Attach the trained adapter

In [ ]:
import glob
import os

from peft import PeftModel

cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert cfgs, (
    "No adapter_config.json found under /kaggle/input — attach the adapter to eval."
)
adapter_dir = os.path.dirname(cfgs[0])
print("adapter:", adapter_dir)
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()
print("adapter loaded")

## 4. Evaluate — boxed accuracy on the holdout

In [ ]:
import json

import torch


def extract_boxed(text):
    marker = "\\boxed{"
    s = text.rfind(marker)
    if s == -1:
        return None
    i = s + len(marker)
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(ch)
        i += 1
    return "".join(out)


def score(pred, gold, tol=1e-2):
    if pred is None:
        return False
    p, g = pred.strip(), str(gold).strip()
    if p == g:
        return True
    try:
        return abs(float(p) - float(g)) <= tol
    except ValueError:
        return False


correct = 0
for k, r in enumerate(holdout):
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(chat, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    gen = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    if score(extract_boxed(gen), r["answer"]):
        correct += 1
    if (k + 1) % 25 == 0:
        print(f"{k + 1}/{len(holdout)} | acc so far {correct / (k + 1):.3f}")

acc = correct / len(holdout)
print(f"\nHOLDOUT boxed accuracy: {acc:.4f} ({correct}/{len(holdout)})")
json.dump(
    {"accuracy": acc, "correct": correct, "n": len(holdout)},
    open("/kaggle/working/holdout_result.json", "w"),
)